# Build an AI teammate for Slack

Tag your bot in Slack to search conversations, investigate projects, analyze data,
or prepare a GitHub pull request. Each Slack thread gets its own Agents API
session and isolated workspace.

```mermaid
sequenceDiagram
    participant Person
    participant Slack
    participant Agent as Agents API
    participant Tools as Slack tools and shared apps
    participant Sandbox as Thread sandbox
    Person->>Slack: @Agent Teammate Investigate this issue
    Slack->>Agent: Create or resume the thread's session
    Agent->>Tools: Search Slack, Notion, Drive, or GitHub
    Agent->>Sandbox: Analyze data or prepare repository changes
    Agent-->>Slack: Stream progress and reply in the thread
    Person->>Slack: @Agent Teammate Open a pull request
    Slack->>Agent: Continue the same session
```


## Agents API capabilities

Sandbox, Persistent sessions, MCP, Vaults and OAuth, Multi-agent, Streaming.

### Every Slack thread keeps its own context

One persistent session per Slack thread remembers earlier questions, findings, and files between follow-ups.

### One bot connects your workplace tools

Bot-scoped Slack tools and an optional shared vault connect Notion, Google Drive, and GitHub without requiring each person to authorize every app.

### A sandbox makes the bot capable

Each thread gets an isolated workspace where the agent can run code, analyze data, inspect repositories, and prepare pull requests.

### Users stay informed while work is running

Session events surface tool activity, specialist handoffs, and answers directly in the original Slack conversation.


## Application flow

1. Slack mention.
2. Slack bot tools.
3. Shared workplace apps.
4. Thread sandbox.
5. Agent progress.
6. Reply + follow-up.


## What you need

- Python 3.14+ and `uv`.
- A sandbox: self-hosted Docker or a [third-party provider](https://developers.openai.com/api/docs/guides/agents-api/environments/self-hosted#sandbox-providers).
- An OpenAI API key and a separate restricted executor key.
- A Slack workspace where you can install an internal application.


## 1. Build the sandbox image

From the repository root:

```bash
cp examples/agents_api/apps/slack_bot/.env.example examples/agents_api/apps/slack_bot/.env
docker build -t agent-api-sandbox:latest examples/agents_api/sandboxes/application_managed/docker
```

Set `OPENAI_API_KEY` and `OPENAI_EXECUTOR_API_KEY` in `examples/agents_api/apps/slack_bot/.env`. Use keys with the same owner, organization, and project. Only the executor key enters the sandbox. It needs `api.agents.environments.connect` and IP restrictions that allow the sandbox's outbound network. The application loads this file automatically.

To create an executor key with the required permission, open [Agents > Environments > Keys](https://platform.openai.com/agents?tab=environments&environment_view=keys) and select **Create**.

For production, you can replace Docker with a hosted [sandbox provider](https://developers.openai.com/api/docs/guides/agents-api/environments/self-hosted#sandbox-providers).


## 2. Create the Slack app

1. Create an app using `examples/agents_api/apps/slack_bot/slack-app-manifest.yaml`.
2. Install it in your workspace and copy the **Bot User OAuth Token**.
3. Create an app-level token with the `connections:write` scope.

Add both tokens to `examples/agents_api/apps/slack_bot/.env`:

```bash
SLACK_BOT_TOKEN=xoxb-...
SLACK_APP_TOKEN=xapp-...
```

Start the bot:

```bash
uv run examples/agents_api/apps/slack_bot/main.py
```

Socket Mode receives messages without a public webhook or OAuth callback. The
bot can read only conversations it belongs to.


## 3. Tag the bot in Slack

Invite the bot to a channel, then mention it:

```text
@Agent Teammate Summarize this week's launch decisions and find the owner.
```

Follow up in the same Slack thread:

```text
Find the related GitHub issue and explain the likely cause.

Reproduce the problem, prepare a fix, and open a pull request.

Only include customer-facing changes.
```

The first message creates a `self_hosted` Agents API session and starts a Docker
container running `codex exec-server`. Replies reuse the same session and
workspace. Send `stop` to cancel an active request.


## Connect shared workplace tools

The bot already has tools for searching the current Slack channel, reading recent
messages, finding teammates, and listing shared files. Add any optional shared
workplace credentials to `.env`:

```bash
NOTION_TOKEN=...
GOOGLE_DRIVE_TOKEN=...
GITHUB_TOKEN=...
```

The application stores configured credentials in one shared vault for the Slack
workspace and connects the corresponding MCP servers:

- Notion: `https://mcp.notion.com/mcp`
- Google Drive: `https://drivemcp.googleapis.com/mcp/v1`
- GitHub: `https://api.githubcopilot.com/mcp/`

Notion and Google Drive require OAuth access tokens for the shared account;
Notion's hosted MCP server does not accept internal integration secrets. GitHub
accepts an appropriately scoped personal access token. Everyone who can use the
bot shares these connections, so grant the account access only to documents and
repositories intended for that audience. Opening a pull request also requires a
GitHub credential with the necessary repository permissions.

### Let Agents API refresh Google Drive access

Obtain a grant through your [Google OAuth app](https://developers.google.com/identity/protocols/oauth2/web-server#offline), requesting offline access and only the Drive scopes your bot needs. Add the grant to `.env`:

```bash
GOOGLE_DRIVE_TOKEN=...
GOOGLE_DRIVE_REFRESH_TOKEN=...
GOOGLE_DRIVE_CLIENT_ID=...
GOOGLE_DRIVE_CLIENT_SECRET=...
GOOGLE_DRIVE_TOKEN_EXPIRES_AT=2026-09-01T12:00:00Z
```

Use the access token's actual expiration, or leave that field empty if unknown. The bot creates a `mcp_oauth` vault credential with Google's token endpoint and refresh configuration. Agents API refreshes the access token when needed; your application owns the initial consent flow.

Restarting the bot reuses the stored OAuth credential instead of replacing refreshed tokens with old `.env` values. If this workspace already has a static Google Drive credential, archive it once before switching to OAuth refresh. Renew or revoke an existing grant through the vault credential APIs; editing `.env` does not replace it.


## Run this notebook

Use a Jupyter Python kernel (Python 3.11 or later) on macOS or Linux in a local clone of the [Cookbook repository](https://github.com/openai/openai-cookbook). The application itself uses Python 3.14; `uv run` installs the dependencies declared in `main.py` and selects that interpreter.

The terminal commands above run the checked-in application. The notebook instead builds a separate copy inside an ignored `tmp_` workspace under your Cookbook checkout. Each `%%writefile` cell contains actual application source. Run these cells in order: the first cell for a module creates its file, and later cells append to it. Python definitions are executed by the application when you launch it.

The setup cell copies only the listed supporting fixtures, manifests, and policy files. It creates a fresh `.env` from the example template without copying your existing credentials. Configure the printed `.env` path before the optional launch step. Rerunning setup creates a new workspace; keep the previous workspace if you need its reports or memory.

Default execution builds and checks the files locally. Docker builds and live API calls require the explicit flags in the launch section.


In [ ]:
from pathlib import Path
import shutil
import tempfile

if globals().get("application_process") is not None and application_process.poll() is None:
    raise RuntimeError("Stop the running application before creating a new workspace.")
application_process = None

# Start Jupyter anywhere inside the Cookbook checkout.
working_directory = Path.cwd().resolve()
cookbook_root = next(
    (path for path in [working_directory, *working_directory.parents]
     if (path / "examples/agents_api/apps/slack_bot/main.py").is_file()),
    None,
)
if cookbook_root is None:
    raise FileNotFoundError("Clone openai/openai-cookbook and start Jupyter inside it.")

notebook_root = Path(tempfile.mkdtemp(prefix="tmp_agents_slack_bot_", dir=cookbook_root))
application_dir = notebook_root / "examples/agents_api/apps/slack_bot"
application_dir.mkdir(parents=True)
for package in [notebook_root / "examples", notebook_root / "examples/agents_api",
                notebook_root / "examples/agents_api/apps", application_dir]:
    (package / "__init__.py").touch()

support_paths = [
    ".env.example",
    "slack-app-manifest.yaml"
]
source_dir = cookbook_root / "examples/agents_api/apps/slack_bot"
for relative_path in support_paths:
    destination = application_dir / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_dir / relative_path, destination)
shutil.copy2(application_dir / ".env.example", application_dir / ".env")
shutil.copytree(
    cookbook_root / "examples/agents_api/sandboxes/application_managed/docker",
    notebook_root / "examples/agents_api/sandboxes/application_managed/docker",
)
print(f"Application workspace: {application_dir}")
print(f"Configure credentials in: {application_dir / '.env'}")


## Implementation walkthrough

This walkthrough connects Slack events to shared tools, one sandbox per thread, and streamed updates.

Follow the setup instructions above, then build the application with the code cells below.


### 1. Set up the bot and sandbox

Clone the repository, copy the example's environment template, and build the Docker image that runs one Codex executor for each Slack thread.


### 2. Connect the Slack bot

Create and install the Slack app using the example manifest. Its bot token reads channels the bot belongs to and posts replies; the app token receives events through Socket Mode.

Socket Mode requires no public webhook, OAuth callback, or individual Slack user tokens.


### 3. Start when someone tags the bot

A Slack mention starts the journey. Use the channel and thread timestamp as the session key, then continue the same Agents API session when teammates follow up.

![A Slack mention connected to a persistent Agents API session.](../../../../images/agents_api/agents-api-slack-bot-message.webp)


### 4. Give the agent scoped Slack tools

Run Slack tools inside your application with the bot token. Bind message and file access to the conversation that triggered the request, keeping the token out of the model and sandbox.

The complete example also reads recent messages, finds teammates, and lists files shared in the current channel.


### 5. Add optional shared workplace credentials

Store shared workplace credentials in a vault. For Google Drive, add an OAuth refresh grant so Agents API can renew access without asking for a new token on every run.

Your Google OAuth app obtains consent and the initial tokens. Add the refresh token, client ID, client secret, and actual access-token expiration to .env. The runnable example reuses existing OAuth credentials without resetting refreshed tokens on restart. Archive an existing static Drive credential before switching auth types. Shared connections must contain only content intended for everyone who can use the bot.


### 6. Connect workplace apps through MCP

Add only the integrations configured for the bot. Slack remains an application tool; Notion, Google Drive, and GitHub are service-connected MCP tools.

Google Drive's hosted MCP server is in developer preview.


### 7. Create a self-hosted session for the thread

A persistent Agents API session owns the conversation, connected tools, and workspace. Specialist agents can divide a larger investigation when needed.

![An agent coordinating Slack, connected workplace tools, and an isolated sandbox.](../../../../images/agents_api/agents-api-slack-bot-investigation.webp)


### 8. Start the thread's sandbox

Launch a Docker container for the session's environment ID. The Codex executor connects back to the Agents API and stays available for later messages in the same Slack thread.


### 9. Stream progress back into Slack

Once the executor is connected, start a turn and update one Slack message as the agent checks MCP tools, delegates research, or prepares a result.


### 10. Keep follow-ups in the same thread

Reuse the original session and sandbox for each follow-up. A message received during an active turn can steer the investigation or cancel it.

Everyone in the thread shares the bot's configured permissions. Repository changes require a GitHub credential with write access.


### 11. Start the bot

Set the bot token and app token, then start receiving Slack events through Socket Mode.


### 12. Try the complete journey in Slack

Invite the bot to a Slack channel and tag it. Continue the conversation in the same thread to research a problem and ask for a real change.


### 13. Clean up when the application shuts down

Keep the thread's session and sandbox alive for follow-ups. The example deletes sessions and stops their Docker containers when the application shuts down. Add an inactive-thread expiration policy when deploying a long-running bot.


## Build the application

The following cells include every application module. Run all cells for each file before launching. The inline dependency declaration in `main.py` installs the current OpenAI SDK and the application libraries through `uv`.


### tools.py

Bind Slack search, messages, users, and files to the conversation that triggered the request.


In [ ]:
%%writefile "{application_dir}/tools.py"
"""Read Slack messages, teammates, and shared files."""

from __future__ import annotations

from typing import Any

from openai.lib.streaming.agents import AsyncToolHandler
from openai.types.beta.agent_tool_param import AgentToolConfigParamFunction
from slack_sdk.web.async_client import AsyncWebClient


def define_tool(
    name: str, description: str, *fields: str
) -> AgentToolConfigParamFunction:
    return {
        "type": "function",
        "name": name,
        "description": description,
        "parameters": {
            "type": "object",
            "properties": {field: {"type": "string"} for field in fields},
            "required": list(fields),
            "additionalProperties": False,
        },
    }


SLACK_TOOLS: list[AgentToolConfigParamFunction] = [
    define_tool(
        "search_slack_messages",
        "Search recent messages in this Slack channel.",
        "query",
    ),
    define_tool("read_slack_channel", "Read recent messages in this Slack channel."),
    define_tool("find_teammate", "Find a Slack teammate by name.", "query"),
    define_tool("list_slack_files", "List files shared in this Slack channel."),
]


def slack_handlers(
    slack: AsyncWebClient, channel_id: str
) -> dict[str, AsyncToolHandler]:
    async def search_slack_messages(arguments: dict[str, Any]) -> dict[str, Any]:
        query = str(arguments["query"]).casefold()
        response = await slack.conversations_history(channel=channel_id, limit=100)
        messages: list[dict[str, Any]] = response.get("messages", [])
        return {
            "messages": [
                {
                    "user": str(message.get("user", "")),
                    "text": str(message.get("text", "")),
                }
                for message in messages
                if query in str(message.get("text", "")).casefold()
            ][:20]
        }

    async def read_slack_channel(_arguments: dict[str, Any]) -> dict[str, Any]:
        response = await slack.conversations_history(channel=channel_id, limit=20)
        messages: list[dict[str, Any]] = response.get("messages", [])
        return {
            "messages": [
                {
                    "user": str(message.get("user", "")),
                    "text": str(message.get("text", "")),
                }
                for message in messages
            ]
        }

    async def find_teammate(arguments: dict[str, Any]) -> dict[str, Any]:
        query = str(arguments["query"]).casefold()
        response = await slack.users_list(limit=200)
        members: list[dict[str, Any]] = response.get("members", [])
        matches: list[dict[str, Any]] = []
        for member in members:
            if member.get("deleted") or member.get("is_bot"):
                continue
            profile = member.get("profile", {})
            name = str(profile.get("display_name") or member.get("real_name") or "")
            if (
                query in name.casefold()
                or query in str(member.get("name", "")).casefold()
            ):
                matches.append({"id": str(member.get("id", "")), "name": name})
        return {"users": matches[:20]}

    async def list_slack_files(_arguments: dict[str, Any]) -> dict[str, Any]:
        response = await slack.files_list(channel=channel_id, count=20)
        files: list[dict[str, Any]] = response.get("files", [])
        return {
            "files": [
                {
                    "name": str(file.get("name", "")),
                    "title": str(file.get("title", "")),
                    "url": str(file.get("permalink", "")),
                }
                for file in files
            ]
        }

    return {
        "search_slack_messages": search_slack_messages,
        "read_slack_channel": read_slack_channel,
        "find_teammate": find_teammate,
        "list_slack_files": list_slack_files,
    }


### connections.py

Reuse shared vault credentials and configure optional workplace MCP connections, including Google OAuth refresh grants.


In [ ]:
%%writefile "{application_dir}/connections.py"
"""Connect workplace MCP tools through vault credentials."""

from __future__ import annotations

import asyncio
import os

from openai import AsyncOpenAI
from openai.types.beta import AgentToolParam
from openai.types.beta.agent_tool_param import AgentToolConfigParamMcp
from openai.types.beta.agents.vaults.credential_auth_create_param import (
    CredentialAuthCreateParam,
)

from .tools import SLACK_TOOLS

NOTION_MCP_URL = "https://mcp.notion.com/mcp"
GOOGLE_DRIVE_MCP_URL = "https://drivemcp.googleapis.com/mcp/v1"
GITHUB_MCP_URL = "https://api.githubcopilot.com/mcp/"


def mcp_server(label: str, url: str) -> AgentToolConfigParamMcp:
    return {
        "type": "mcp",
        "server_label": label,
        "transport": {"type": "http", "server_url": url},
        "connection_origin": "service",
    }


class Connections:
    """Configure workplace tools once per Slack workspace."""

    def __init__(self, client: AsyncOpenAI) -> None:
        self.client = client
        self.vaults: dict[str, str] = {}
        self.lock = asyncio.Lock()

    async def get(self, team_id: str) -> tuple[str | None, list[AgentToolParam]]:
        async with self.lock:
            return await self._configure_tools(team_id)

    async def _configure_tools(
        self, team_id: str
    ) -> tuple[str | None, list[AgentToolParam]]:
        notion_token = os.environ.get("NOTION_TOKEN")
        google_drive_token = os.environ.get("GOOGLE_DRIVE_TOKEN")
        github_token = os.environ.get("GITHUB_TOKEN")
        google_refresh_token = os.environ.get("GOOGLE_DRIVE_REFRESH_TOKEN")
        if google_refresh_token:
            for name in (
                "GOOGLE_DRIVE_TOKEN",
                "GOOGLE_DRIVE_CLIENT_ID",
                "GOOGLE_DRIVE_CLIENT_SECRET",
            ):
                if not os.environ.get(name):
                    raise ValueError(
                        f"Set {name} when enabling Google Drive OAuth refresh."
                    )
        tools: list[AgentToolParam] = [*SLACK_TOOLS, {"type": "web_search"}]
        if notion_token:
            tools.append(mcp_server("notion", NOTION_MCP_URL))
        if google_drive_token:
            tools.append(mcp_server("google_drive", GOOGLE_DRIVE_MCP_URL))
        if github_token:
            tools.append(mcp_server("github", GITHUB_MCP_URL))
        if not (notion_token or google_drive_token or github_token):
            return None, tools

        vault_id = self.vaults.get(team_id)
        if vault_id is not None:
            return vault_id, tools

        existing = [
            candidate
            async for candidate in self.client.beta.agents.vaults.list(limit=100)
        ]
        vault = next(
            (
                candidate
                for candidate in existing
                if candidate.metadata.get("slack_team_id") == team_id
                and candidate.metadata.get("owner") == "slack_bot"
            ),
            None,
        )
        if vault is None:
            vault = await self.client.beta.agents.vaults.create(
                name="Slack teammate integrations",
                metadata={"slack_team_id": team_id, "owner": "slack_bot"},
            )
        credentials = [
            item
            async for item in self.client.beta.agents.vaults.credentials.list(vault.id)
        ]

        async def save_credential(label: str, auth: CredentialAuthCreateParam) -> None:
            credential = next(
                (
                    item
                    for item in credentials
                    if item.auth.mcp_server_url == auth["mcp_server_url"]
                ),
                None,
            )
            if credential is not None and credential.auth.type != auth["type"]:
                raise ValueError(
                    f"Archive the existing {label} vault credential before changing auth type."
                )
            if credential is None:
                await self.client.beta.agents.vaults.credentials.create(
                    vault.id,
                    name=label.replace("_", " ").title(),
                    auth=auth,
                )
            elif auth["type"] == "static_bearer":
                await self.client.beta.agents.vaults.credentials.update(
                    credential.id,
                    vault_id=vault.id,
                    auth={"type": "static_bearer", "token": auth["token"]},
                )
            # Keep refreshed OAuth grants in the vault; .env only seeds a new credential.

        if notion_token:
            await save_credential(
                "notion",
                {
                    "type": "static_bearer",
                    "mcp_server_url": NOTION_MCP_URL,
                    "token": notion_token,
                },
            )

        if google_drive_token:
            google_auth: CredentialAuthCreateParam = {
                "type": "static_bearer",
                "mcp_server_url": GOOGLE_DRIVE_MCP_URL,
                "token": google_drive_token,
            }
            if google_refresh_token:
                google_auth = {
                    "type": "mcp_oauth",
                    "mcp_server_url": GOOGLE_DRIVE_MCP_URL,
                    "access_token": google_drive_token,
                    "expires_at": os.environ.get("GOOGLE_DRIVE_TOKEN_EXPIRES_AT")
                    or None,
                    "refresh": {
                        "token_endpoint": "https://oauth2.googleapis.com/token",
                        "client_id": os.environ["GOOGLE_DRIVE_CLIENT_ID"],
                        "refresh_token": google_refresh_token,
                        "token_endpoint_auth": {
                            "type": "client_secret_post",
                            "client_secret": os.environ["GOOGLE_DRIVE_CLIENT_SECRET"],
                        },
                    },
                }
            await save_credential("google_drive", google_auth)

        if github_token:
            await save_credential(
                "github",
                {
                    "type": "static_bearer",
                    "mcp_server_url": GITHUB_MCP_URL,
                    "token": github_token,
                },
            )

        self.vaults[team_id] = vault.id
        return vault.id, tools


### agent.py

Create or resume a session per thread, start its executor, stream progress, and handle steering, cancellation, and shutdown.


In [ ]:
%%writefile "{application_dir}/agent.py"
"""Run and reuse one Agents API sandbox per Slack thread."""

from __future__ import annotations

import asyncio
import os
import sys
from collections.abc import AsyncIterator, Awaitable, Callable

import docker
from docker.models.containers import Container
from openai import AsyncOpenAI, NotFoundError
from openai.types.beta import AgentSessionEvent
from openai.types.beta.agents.session_create_params import Agent
from slack_sdk.web.async_client import AsyncWebClient

from .connections import Connections
from .tools import slack_handlers

INSTRUCTIONS = """\
You are a capable Slack teammate.
Search the current Slack conversation and connected Notion, Google Drive, and GitHub sources.
Cite the records behind your answer.
Use your workspace to analyze data, inspect repositories, and prepare changes.
Only modify external systems or create pull requests when the user explicitly asks.
Never reveal private information in a shared channel.
Delegate complex investigations when helpful.
"""
ProgressCallback = Callable[[str], Awaitable[None]]


def start_executor(environment_id: str, remote_url: str) -> Container:
    return docker.from_env().containers.run(
        os.environ.get("AGENTS_SANDBOX_IMAGE", "agent-api-sandbox:latest"),
        [
            "codex",
            "exec-server",
            "--remote",
            remote_url,
            "--environment-id",
            environment_id,
        ],
        environment={"CODEX_API_KEY": os.environ["OPENAI_EXECUTOR_API_KEY"]},
        detach=True,
        auto_remove=True,
        init=True,
    )


class SlackBot:
    """Keep one self-hosted Agents API session and sandbox per Slack thread."""

    def __init__(self, client: AsyncOpenAI, slack: AsyncWebClient) -> None:
        self.client = client
        self.slack = slack
        self.sessions: dict[str, str] = {}
        self.sandboxes: dict[str, Container] = {}
        self.connections = Connections(client)
        self.model = os.environ.get("OPENAI_MODEL", "gpt-5.6-sol")
        self.active: set[str] = set()
        self.seen_messages: set[str] = set()
        self.thread_locks: dict[str, asyncio.Lock] = {}

    async def answer(
        self,
        question: str,
        *,
        thread_id: str,
        team_id: str,
        channel_id: str,
        on_progress: ProgressCallback | None = None,
    ) -> str:
        async with self.thread_locks.setdefault(thread_id, asyncio.Lock()):
            return await self._answer(
                question, thread_id, team_id, channel_id, on_progress
            )

    async def _answer(
        self,
        question: str,
        thread_id: str,
        team_id: str,
        channel_id: str,
        on_progress: ProgressCallback | None,
    ) -> str:
        if thread_id in self.sessions:
            session = await self.client.beta.agents.sessions.retrieve(
                self.sessions[thread_id]
            )
        else:
            vault_id, tools = await self.connections.get(team_id)
            agent: Agent = {
                "model": self.model,
                "instructions": INSTRUCTIONS,
                "reasoning": {"effort": "medium"},
                "multi_agent": {"enabled": True, "max_concurrent_subagents": 3},
                "tools": tools,
            }
            session = await self.client.beta.agents.sessions.create(
                agent=agent,
                environment={
                    "type": "self_hosted",
                    "workspace_directory": "/workspace",
                },
                vault_ids=[vault_id] if vault_id is not None else None,
            )

            try:
                environment = session.environment
                if environment.type != "self_hosted":
                    raise RuntimeError("Expected a self-hosted execution environment.")
                sandbox = await asyncio.to_thread(
                    start_executor, environment.id, environment.remote_url
                )
            except BaseException as original_error:
                try:
                    await self.client.beta.agents.sessions.delete(session.id)
                except Exception as cleanup_error:
                    original_error.add_note(
                        f"Could not delete session {session.id}: {cleanup_error}"
                    )
                raise

            self.sessions[thread_id] = session.id
            self.sandboxes[thread_id] = sandbox

        async with self.client.beta.agents.sessions.stream(
            session.id,
            input=question,
            tool_handlers=slack_handlers(self.slack, channel_id),
        ) as events:
            return await self._collect_reply(events, thread_id, on_progress)



Continue `agent.py`: `SlackBot._collect_reply`, `SlackBot.steer`, `SlackBot.cancel`, `SlackBot.close`. This cell appends to the same file.


In [ ]:
%%writefile -a "{application_dir}/agent.py"
    async def _collect_reply(
        self,
        events: AsyncIterator[AgentSessionEvent],
        thread_id: str,
        on_progress: ProgressCallback | None,
    ) -> str:
        parts: list[str] = []
        last_progress = ""
        self.active.add(thread_id)

        try:
            async for event in events:
                if (
                    event.type == "agent.session.turn.item.added"
                    and event.item is not None
                ):
                    item = event.item.to_dict()
                    label = str(item.get("server_label") or item.get("name", ""))
                    progress = {
                        "search_slack_messages": "Searching Slack conversations...",
                        "read_slack_channel": "Reading the Slack channel...",
                        "find_teammate": "Looking up a teammate...",
                        "list_slack_files": "Checking shared files...",
                        "notion": "Checking Notion...",
                        "google_drive": "Searching Google Drive...",
                        "github": "Checking GitHub...",
                    }.get(label, "Working on your request...")
                    if on_progress is not None and progress != last_progress:
                        await on_progress(progress)
                        last_progress = progress
                elif (
                    event.type == "agent.session.subagent.created"
                    and on_progress is not None
                ):
                    await on_progress("A specialist is investigating...")

                if event.type == "agent.session.turn.output_text.delta":
                    parts.append(event.delta)
                elif event.type == "agent.session.turn.output_text.done" and not parts:
                    parts.append(event.text)
                elif event.type in {
                    "agent.session.failed",
                    "agent.session.turn.failed",
                    "error",
                }:
                    raise RuntimeError(f"Agent session failed: {event.to_dict()}")
                elif event.type == "agent.session.turn.cancelled":
                    return "Request cancelled."
        finally:
            self.active.discard(thread_id)

        return "".join(parts)

    async def steer(self, thread_id: str, instructions: str) -> None:
        await self.client.beta.agents.sessions.events.create(
            self.sessions[thread_id],
            events=[
                {
                    "type": "agent.session.input.message",
                    "input": [
                        {
                            "role": "user",
                            "content": [{"type": "input_text", "text": instructions}],
                        }
                    ],
                }
            ],
        )

    async def cancel(self, thread_id: str) -> None:
        await self.client.beta.agents.sessions.events.create(
            self.sessions[thread_id], events=[{"type": "agent.session.input.cancel"}]
        )

    async def close(self) -> None:
        original_error = sys.exception()
        errors: list[Exception] = []
        for thread_id, session_id in self.sessions.items():
            try:
                await self.client.beta.agents.sessions.delete(session_id)
            except NotFoundError:
                pass
            except Exception as error:
                error.add_note(f"Could not delete session {session_id}.")
                if original_error is not None:
                    original_error.add_note(
                        f"Could not delete session {session_id}: {error}"
                    )
                errors.append(error)
            sandbox = self.sandboxes.get(thread_id)
            if sandbox is not None:
                try:
                    await asyncio.to_thread(sandbox.remove, force=True)
                except docker.errors.NotFound:
                    pass
                except Exception as error:
                    error.add_note(
                        f"Could not remove sandbox for session {session_id}."
                    )
                    if original_error is not None:
                        original_error.add_note(
                            f"Could not remove sandbox for session {session_id}: {error}"
                        )
                    errors.append(error)
        if errors and original_error is None:
            raise ExceptionGroup("Could not close all Slack runtimes", errors)
        if not errors:
            self.sessions.clear()
            self.sandboxes.clear()


### main.py

Connect Slack Socket Mode events to the bot and keep replies in the original thread.


In [ ]:
%%writefile "{application_dir}/main.py"
# /// script
# requires-python = ">=3.14"
# dependencies = [
#     "openai>=3.13.0",
#     "aiohttp",
#     "docker",
#     "python-dotenv",
#     "slack-bolt",
# ]
# ///

"""Start the Slack bot with Socket Mode."""

from __future__ import annotations

import asyncio
import logging
import os
import sys
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from openai import AsyncOpenAI

# Support direct execution from any working directory.
if __package__ in {None, ""}:
    sys.path.insert(0, str(Path(__file__).resolve().parents[4]))


from examples.agents_api.apps.slack_bot.agent import SlackBot

EXAMPLE_DIR = Path(__file__).resolve().parent


async def run_slack() -> None:
    from slack_bolt.adapter.socket_mode.aiohttp import AsyncSocketModeHandler
    from slack_bolt.async_app import AsyncApp
    from slack_bolt.context.async_context import AsyncBoltContext
    from slack_bolt.context.say.async_say import AsyncSay

    app = AsyncApp(token=os.environ["SLACK_BOT_TOKEN"])

    async with AsyncOpenAI() as client:
        bot = SlackBot(client, app.client)

        async def handle_message(
            event: dict[str, Any], context: AsyncBoltContext, say: AsyncSay
        ) -> None:
            if event.get("bot_id"):
                return

            question = (
                str(event.get("text", ""))
                .replace(f"<@{context.bot_user_id}>", "")
                .strip()
            )
            if not question:
                return

            channel = str(event["channel"])
            thread_ts = str(event.get("thread_ts") or event["ts"])
            team_id = str(context.team_id or event.get("team", ""))
            thread_id = f"{team_id}:{channel}:{thread_ts}"
            message_id = f"{team_id}:{channel}:{event['ts']}"
            if message_id in bot.seen_messages:
                return
            bot.seen_messages.add(message_id)

            if thread_id in bot.active:
                if question.lower() in {"stop", "cancel", "stop this", "cancel this"}:
                    await bot.cancel(thread_id)
                    await say(text="Stopping your request.", thread_ts=thread_ts)
                else:
                    await bot.steer(thread_id, question)
                    await say(
                        text="Got it. Updating your request.", thread_ts=thread_ts
                    )
                return

            status = await say(text="Checking your request...", thread_ts=thread_ts)
            status_ts = str(status["ts"])

            async def update_progress(message: str) -> None:
                await context.client.chat_update(
                    channel=channel, ts=status_ts, text=message
                )

            try:
                reply = await bot.answer(
                    question,
                    thread_id=thread_id,
                    team_id=team_id,
                    channel_id=channel,
                    on_progress=update_progress,
                )
            except Exception:
                logging.exception("Slack request failed for thread %s", thread_id)
                await context.client.chat_update(
                    channel=channel,
                    ts=status_ts,
                    text="I could not complete that request. Check the bot's logs and try again.",
                )
                return

            await context.client.chat_update(channel=channel, ts=status_ts, text=reply)

        @app.event("app_mention")
        async def reply_to_mention(
            event: dict[str, Any], context: AsyncBoltContext, say: AsyncSay
        ) -> None:
            await handle_message(event, context, say)

        @app.event("message")
        async def reply_to_message(
            event: dict[str, Any], context: AsyncBoltContext, say: AsyncSay
        ) -> None:
            if event.get("channel_type") != "im":
                thread_ts = event.get("thread_ts")
                if thread_ts is None:
                    return
                thread_id = f"{context.team_id}:{event['channel']}:{thread_ts}"
                if thread_id not in bot.sessions:
                    return
            await handle_message(event, context, say)

        try:
            await AsyncSocketModeHandler(
                app, os.environ["SLACK_APP_TOKEN"]
            ).start_async()
        finally:
            await bot.close()




Continue `main.py`: `main`, `Application entry point`. This cell appends to the same file.


In [ ]:
%%writefile -a "{application_dir}/main.py"
def main() -> None:
    load_dotenv(EXAMPLE_DIR / ".env")
    asyncio.run(run_slack())


if __name__ == "__main__":
    main()


## Check the generated files

Compile all generated modules without importing them or contacting external services. This catches syntax errors before you launch the application.


In [ ]:
import py_compile

modules = ["tools.py", "connections.py", "agent.py", "main.py"]
for filename in modules:
    py_compile.compile(str(application_dir / filename), doraise=True)
print(f"Compiled {len(modules)} application modules.")


## Launch the application (optional)

Edit the generated `.env` file with the credentials listed above. Install `uv` and, for sandbox applications, start Docker. The following cells are disabled by default. Enabling them may incur API usage and connect to the configured services.

Use the generated workspace for every path below. For a hosted deployment, package the generated application files and supply credentials through your deployment's secret configuration.


In [ ]:
import subprocess

BUILD_SANDBOX = False
if BUILD_SANDBOX:
    subprocess.run(
        ["docker", "build", "-t", "agent-api-sandbox:latest", str(notebook_root / "examples/agents_api/sandboxes/application_managed/docker")],
        cwd=notebook_root,
        check=True,
    )


This application stays running to receive events. The process writes to `application.log` in the generated workspace. Inspect that file for startup failures and progress. Use the stop cell below when you finish.


In [ ]:
import os
import subprocess

RUN_APPLICATION = False
application_arguments = []
if RUN_APPLICATION:
    if application_process is not None and application_process.poll() is None:
        raise RuntimeError("Stop the previous application before launching again.")
    with (application_dir / "application.log").open("w") as application_log:
        application_process = subprocess.Popen(
            ["uv", "run", str(application_dir / "main.py"), *application_arguments],
            cwd=notebook_root,
            env={key: value for key, value in os.environ.items() if key != "VIRTUAL_ENV"},
            start_new_session=True,
            stdout=application_log,
            stderr=subprocess.STDOUT,
        )
    print(f"Process started: {application_process.pid}")
    print(f"Progress log: {application_dir / 'application.log'}")


### Stop a running application

Set `STOP_APPLICATION = True` after you finish. Interrupt the application process group so the application's shutdown handlers can close sessions and remove containers. Batch commands normally exit on their own. A timeout means shutdown is still in progress; inspect the log before taking further action.


In [ ]:
import os
import signal

STOP_APPLICATION = False
if STOP_APPLICATION and application_process is not None:
    if application_process.poll() is None:
        os.killpg(os.getpgid(application_process.pid), signal.SIGINT)
        application_process.wait(timeout=30)
    print(f"Application exited with status {application_process.returncode}.")


The generated workspace remains available for reports and memory. Remove it manually after stopping the application and saving any files you need. Do not rerun the launch cell while the previous process is running.


## Example result

A Slack mention now starts a persistent investigation using shared workplace tools and an isolated workspace for completing the task.

The following illustrates a possible result; model-generated findings depend on the inputs and connected sources.

```text
You: @Agent Teammate What is blocking the Phoenix launch?
Bot: Searching Slack conversations...
Bot: Checking GitHub...
Bot: The checkout accessibility issue is blocking sign-off.
     Priya owns the launch and Maya is reviewing the fix.

You: Reproduce the issue and prepare a patch.
Bot: Checking the repository and running the relevant tests...
Bot: I reproduced the missing focus state and prepared a fix.

You: Open a pull request.
Bot: Opened a pull request with the fix and test coverage.
```


## Next steps

- Connect Notion, Google Drive, or GitHub using a dedicated shared account with limited access.
- Persist thread-to-session mappings so conversations survive application restarts.
- Replace local Docker with your preferred hosted sandbox provider.


## Related documentation

- [Sandbox providers](https://developers.openai.com/api/docs/guides/agents-api/environments/self-hosted#sandbox-providers): Choose a local or hosted sandbox provider for your agent's isolated workspace.


## Files

- [main.py](https://github.com/openai/openai-cookbook/blob/main/examples/agents_api/apps/slack_bot/main.py): Slack events and application startup.
- [agent.py](https://github.com/openai/openai-cookbook/blob/main/examples/agents_api/apps/slack_bot/agent.py): Agent sessions, sandboxes, streaming, and cleanup.
- [tools.py](https://github.com/openai/openai-cookbook/blob/main/examples/agents_api/apps/slack_bot/tools.py): Slack search, channel, teammate, and file tools.
- [connections.py](https://github.com/openai/openai-cookbook/blob/main/examples/agents_api/apps/slack_bot/connections.py): MCP connections and vault credentials.
